# Credit Card Fraud Analytics — Data Understanding

## Phase 1: Data Understanding & Data Quality

This notebook performs the initial data understanding and quality assessment of the Credit Card Fraud Detection dataset.

**Objectives**
- Understand dataset structure and dimensions
- Inspect data types and missing values
- Identify duplicate records
- Measure class imbalance
- Analyze transaction amounts
- Compare Normal vs Fraud transaction amounts
- Inspect the transaction time range
- Document initial data-quality findings

**Important:** No rows are removed or modified in this notebook. Cleaning decisions are deferred to the dedicated data-cleaning phase.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. Load Dataset

In [ ]:
# Update this path if the notebook is moved to another folder.
DATA_PATH = '../data/creditcard.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 3. Basic Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
display(df.head())

### Initial observation

The dataset contains transaction-level records. `Class` is the target variable: `0` = legitimate transaction and `1` = fraudulent transaction.

`V1`–`V28` are anonymized PCA-transformed features, so their original business meaning is not available.

## 4. Data Types and Memory Usage

In [ ]:
df.info()

In [ ]:
dtype_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null_count': df.notna().sum(),
    'unique_values': df.nunique(),
})
display(dtype_summary)

## 5. Missing Values

In [ ]:
missing_summary = (
    df.isna().sum().to_frame('missing_count')
      .assign(missing_pct=lambda x: x['missing_count'] / len(df) * 100)
      .query('missing_count > 0')
      .sort_values('missing_count', ascending=False)
)

if missing_summary.empty:
    print('No missing values were found in the dataset.')
else:
    display(missing_summary)

## 6. Duplicate Records

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_pct = duplicate_count / len(df) * 100

print(f'Duplicate rows: {duplicate_count:,}')
print(f'Duplicate percentage: {duplicate_pct:.4f}%')

### Interpretation

Duplicate rows exist. They are **not removed in Phase 1** because an identical transaction record does not automatically prove that the transaction is invalid. The cleaning phase will investigate them before deciding whether to remove them.

## 7. Class Distribution

In [ ]:
class_summary = (
    df['Class'].value_counts().rename_axis('Class').to_frame('transaction_count')
)
class_summary['percentage'] = class_summary['transaction_count'] / len(df) * 100
class_summary['label'] = class_summary.index.map({0: 'Normal', 1: 'Fraud'})
display(class_summary[['label', 'transaction_count', 'percentage']])

In [ ]:
fraud_count = (df['Class'] == 1).sum()
normal_count = (df['Class'] == 0).sum()
fraud_rate = fraud_count / len(df) * 100

print(f'Normal transactions: {normal_count:,}')
print(f'Fraud transactions: {fraud_count:,}')
print(f'Fraud rate: {fraud_rate:.4f}%')

### Interpretation

The target variable is highly imbalanced: fraudulent transactions represent only a very small fraction of all transactions.

Therefore, **accuracy alone would be misleading** when evaluating a future fraud-detection model. Precision, Recall, F1-score and PR-AUC will be more informative.

## 8. Transaction Amount Analysis

In [ ]:
amount_summary = df['Amount'].describe()
display(amount_summary.to_frame('Amount'))

In [ ]:
amount_metrics = pd.Series({
    'total_transaction_amount': df['Amount'].sum(),
    'mean_transaction_amount': df['Amount'].mean(),
    'median_transaction_amount': df['Amount'].median(),
    'minimum_transaction_amount': df['Amount'].min(),
    'maximum_transaction_amount': df['Amount'].max(),
    'zero_amount_transactions': (df['Amount'] == 0).sum(),
    'zero_amount_percentage': (df['Amount'] == 0).mean() * 100,
})
display(amount_metrics.to_frame('value'))

### Interpretation

The transaction amount distribution is strongly right-skewed: the mean is substantially higher than the median. A relatively small number of high-value transactions pull the average upward.

For later analysis, median and percentile-based measures should be considered alongside the mean.

## 9. Normal vs Fraud Transaction Amount

In [ ]:
amount_by_class = (
    df.groupby('Class')['Amount']
      .agg(['count', 'sum', 'mean', 'median', 'min', 'max'])
      .rename(index={0: 'Normal', 1: 'Fraud'})
)
amount_by_class['share_of_total_amount_pct'] = amount_by_class['sum'] / df['Amount'].sum() * 100
display(amount_by_class)

### Interpretation

Mean and median should be compared together. Fraudulent transactions are not necessarily high-value transactions, so the full transaction-amount distribution should be investigated during EDA.

## 10. Transaction Time Range

In [ ]:
time_min = df['Time'].min()
time_max = df['Time'].max()
time_span_hours = (time_max - time_min) / 3600

time_summary = pd.Series({
    'minimum_time_seconds': time_min,
    'maximum_time_seconds': time_max,
    'time_span_hours': time_span_hours,
    'time_span_days': time_span_hours / 24,
})
display(time_summary.to_frame('value'))

### Interpretation

`Time` represents elapsed seconds from the beginning of the data collection period. The observed range covers approximately two days.

Later analysis should therefore focus on time-of-day and short-period patterns rather than long-term monthly or seasonal trends.

## 11. Descriptive Statistics

In [ ]:
display(df.describe().T)

### Note on V1–V28

The `V1`–`V28` variables are anonymized PCA-transformed features. Their distributions can be analyzed, but their original business meaning is not available.

## 12. Initial Data Quality Summary

In [ ]:
quality_summary = pd.DataFrame({
    'Metric': [
        'Rows', 'Columns', 'Missing values', 'Duplicate rows',
        'Normal transactions', 'Fraud transactions', 'Fraud rate (%)',
        'Total transaction amount', 'Average transaction amount',
        'Median transaction amount', 'Zero-amount transactions',
        'Minimum Time (seconds)', 'Maximum Time (seconds)'
    ],
    'Value': [
        len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum()),
        int(normal_count), int(fraud_count), fraud_rate, df['Amount'].sum(),
        df['Amount'].mean(), df['Amount'].median(), int((df['Amount'] == 0).sum()),
        df['Time'].min(), df['Time'].max()
    ]
})
display(quality_summary)

## 13. Phase 1 Conclusions

### Key findings
1. The dataset contains 284,807 transaction records and 31 columns.
2. There are no missing values.
3. Duplicate rows are present and require investigation before removal.
4. Fraudulent transactions are extremely rare, creating severe class imbalance.
5. Transaction amounts are strongly right-skewed, so median and percentile measures are important.
6. Fraud and Normal transactions show different transaction-amount distributions that require deeper EDA.
7. The transaction time range is approximately two days, making time-of-day analysis more appropriate than long-term seasonality analysis.
8. V1–V28 are anonymized PCA features and should not be assigned unsupported business meanings.
